# Mechanistically Validated Difficulty Features — MVE

Self-contained. Vendors the `mechanistic-conformal` package, loads the LTSF benchmark data, trains iTransformer + TopK SAE, runs correlation screening + causal ablation, builds the three matched-budget feature sets (SAE-random / SAE-correlated / MVDF), calibrates with Variant B (Mondrian + per-bin ACI) and Variant A (opaque MLP), and reports the RQ1 diagnostic, the first-order `corr(hidden, |e|)` screen, and the H2 / H3 tables per dataset.

**Kaggle setup:**
1. Sidebar → Accelerator → **GPU** (T4 or P100).
2. Sidebar → Internet → **On** (for the ETT download).
3. Sidebar → **Add Input** → attach the multivariate-TS bundle you have (the one with `archive/ETTh.csv`, `ETTm.csv`, `LD2011_2014.csv`, `PeMS.csv`, `mpi_roof.csv`, `solar_Alabama.csv`, `exchange.csv`, `QPS.csv`). The adapters in `data/datasets.py` know these filenames/formats and normalize each to the standard layout (raw electricity/weather are resampled to hourly). Any already-standard `traffic.csv` / `electricity.csv` / `weather.csv` is used directly if present. File discovery walks all of `/kaggle/input` recursively.


## 1. Vendor the package

In [ ]:
import os, base64, json
PKG = 'mechanistic_conformal'
FILES = json.loads(base64.b64decode('...').decode())
for rel, content in FILES.items():
    path = os.path.join(PKG, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, 'w').write(content)
print('wrote', len(FILES), 'files into', PKG)


## 2. Dependencies

In [ ]:
import importlib, subprocess, sys
for pkg in ['pyyaml', 'scipy', 'scikit-learn']:
    name = {'pyyaml':'yaml','scikit-learn':'sklearn'}.get(pkg, pkg)
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


## 2b. Probe the attached inputs

Confirms which datasets the adapters can build before the long run.

In [ ]:
import sys, os
sys.path.insert(0, 'mechanistic_conformal')
os.chdir('mechanistic_conformal')
from data.probe_inputs import probe
probe()
os.chdir('..')


## 3. Configure & run

`device: cuda` if a GPU is attached. Single seed. Full training budget (20-epoch early-stopped iTransformer, 20k SAE steps, 400-feature ablation sweep on up to 2048 windows).

Datasets whose CSVs are not found under `/kaggle/input` (ETT excepted — HTTP) print `SKIP` and are passed over.

In [ ]:
import os, sys, yaml, torch
if 'mechanistic_conformal' not in sys.path: sys.path.insert(0, 'mechanistic_conformal')
if os.path.basename(os.getcwd()) != 'mechanistic_conformal': os.chdir('mechanistic_conformal')
from data.datasets import download

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
base = yaml.safe_load(open('configs/mve_etth1.yaml'))
base['device'] = dev

targets = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'traffic', 'electricity', 'weather', 'solar', 'exchange']
seeds = [0]
available = []
for ds in targets:
    try:
        download(ds); available.append(ds)
    except Exception as e:
        print(f'SKIP {ds}: {e}')
print('will run:', available, 'x seeds', seeds)

import runpy
for ds in available:
    for sd in seeds:
        cfg = dict(base)
        cfg['dataset'] = dict(base['dataset'], name=ds)
        cfg['seed'] = sd
        cfg['run_name'] = f'mve_{ds}_seed{sd}' if len(seeds) > 1 else f'mve_{ds}'
        cpath = f'configs/_run_{ds}_s{sd}.yaml'
        yaml.safe_dump(cfg, open(cpath, 'w'))
        print('=' * 70, f'\n {ds}  seed {sd}\n', '=' * 70)
        sys.argv = ['mve_real', '--config', cpath]
        try:
            runpy.run_module('experiments.mve_real', run_name='__main__')
        except SystemExit:
            pass
        except Exception as e:
            print(f'RUN FAILED {ds} seed {sd}: {type(e).__name__}: {e}')


## 4. Show results + aggregate across seeds

In [ ]:
import json, glob, re, os
import numpy as np, pandas as pd
from IPython.display import display, Image

rows = []
for rd in sorted(glob.glob('runs/mve_*')):
    try:
        log = json.load(open(f'{rd}/run_log.json'))
        rq1 = json.load(open(f'{rd}/rq1_diagnostic.json'))
    except FileNotFoundError:
        continue
    m = re.match(r'runs/mve_(.+?)(?:_seed(\d+))?$', rd)
    ds, seed = m.group(1), int(m.group(2) or 0)
    h3 = log.get('H3_delta_shift_correlated_minus_mvdf', {})
    hd = log.get('hidden_difficulty_signal', {})
    rows.append(dict(dataset=ds, seed=seed,
        fc_val_mse=log.get('forecaster_val_mse'),
        sae_fvu=(log.get('sae_reconstruction') or {}).get('fvu_standardized'),
        corr_hidden_e=hd.get('max_abs_corr_hiddenPC_vs_score'),
        rq1_rho=rq1.get('spearman_rho'), rq1_jaccard=rq1.get('top_set_jaccard'),
        delta_pos_rate=log.get('ablation_positive_rate'),
        h3_d_coverage=h3.get('d_coverage'), h3_d_width=h3.get('d_width'),
        h3_d_winkler=h3.get('d_winkler'), h3_d_mace=h3.get('d_mace')))
raw = pd.DataFrame(rows).sort_values(['dataset','seed'])
print('=== per-run ==='); display(raw)

agg = raw.groupby('dataset').agg(['mean','std']).round(4)
print('=== aggregated across seeds (mean / std) ==='); display(agg)
raw.to_csv('runs/AGG_per_run.csv', index=False)
agg.to_csv('runs/AGG_by_dataset.csv')

# H3 read: positive d_coverage AND negative d_width => MVDF both covers better and
# stays tighter under shift (outcome A). Flag datasets where the sign is stable.
for ds, g in raw.groupby('dataset'):
    cov = g['h3_d_coverage']; wid = g['h3_d_width']
    stable_cov = (cov > 0).all() or (cov < 0).all()
    print(f'{ds:12s} H3 d_coverage {cov.mean():+.4f}+-{cov.std():.4f} '
          f'd_width {wid.mean():+.4f}+-{wid.std():.4f}  '
          f'{"[stable sign]" if stable_cov else "[sign flips across seeds]"}')

for rd in sorted(glob.glob('runs/mve_*')):
    p = f'{rd}/coverage_width_over_time.png'
    if os.path.exists(p):
        print(rd); display(Image(p))


## 5. Zip for download

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/mve_runs', 'zip', 'runs')
print('runs -> /kaggle/working/mve_runs.zip')
